In [7]:
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
DATA_ROOT = PROJECT_ROOT / "data"

print(PROJECT_ROOT)
print(DATA_ROOT)

/home/pasha/projects/bybit-history-downloader
/home/pasha/projects/bybit-history-downloader/data


In [8]:
from marketforge.processing.archives import (
    scan_raw_archives,
    select_raw_archives,
)


archives = scan_raw_archives(
    data_root=DATA_ROOT,
)

selected = select_raw_archives(
    archives,
    exchange="bybit",
    instrument_type="perpetual",
    market_category="linear",
    data_type="trade_ticks",
    symbol="BTCUSDT",
)

print(f"Selected archives: {len(selected)}")

for archive in selected[:10]:
    print(
        archive.start_date,
        archive.end_date,
        archive.path,
    )

Selected archives: 7
2026-08-10 2026-08-10 /home/pasha/projects/bybit-history-downloader/data/raw/bybit/perpetual/linear/trade_ticks/BTCUSDT/BTCUSDT2026-08-10.csv.gz
2026-08-11 2026-08-11 /home/pasha/projects/bybit-history-downloader/data/raw/bybit/perpetual/linear/trade_ticks/BTCUSDT/BTCUSDT2026-08-11.csv.gz
2026-08-12 2026-08-12 /home/pasha/projects/bybit-history-downloader/data/raw/bybit/perpetual/linear/trade_ticks/BTCUSDT/BTCUSDT2026-08-12.csv.gz
2026-08-13 2026-08-13 /home/pasha/projects/bybit-history-downloader/data/raw/bybit/perpetual/linear/trade_ticks/BTCUSDT/BTCUSDT2026-08-13.csv.gz
2026-08-14 2026-08-14 /home/pasha/projects/bybit-history-downloader/data/raw/bybit/perpetual/linear/trade_ticks/BTCUSDT/BTCUSDT2026-08-14.csv.gz
2026-09-02 2026-09-02 /home/pasha/projects/bybit-history-downloader/data/raw/bybit/perpetual/linear/trade_ticks/BTCUSDT/BTCUSDT2026-09-02.csv.gz
2026-09-03 2026-09-03 /home/pasha/projects/bybit-history-downloader/data/raw/bybit/perpetual/linear/trade_tic

In [9]:
from marketforge.processing.archives import (
    scan_raw_archives,
    select_raw_archives,
)


archives = scan_raw_archives(
    data_root=DATA_ROOT,
)

selected = select_raw_archives(
    archives,
    exchange="bybit",
    instrument_type="perpetual",
    market_category="linear",
    data_type="trade_ticks",
    symbol="BTCUSDT",
)

print(f"Selected archives: {len(selected)}")

for archive in selected[:10]:
    print(
        archive.start_date,
        archive.end_date,
        archive.path,
    )

Selected archives: 7
2026-08-10 2026-08-10 /home/pasha/projects/bybit-history-downloader/data/raw/bybit/perpetual/linear/trade_ticks/BTCUSDT/BTCUSDT2026-08-10.csv.gz
2026-08-11 2026-08-11 /home/pasha/projects/bybit-history-downloader/data/raw/bybit/perpetual/linear/trade_ticks/BTCUSDT/BTCUSDT2026-08-11.csv.gz
2026-08-12 2026-08-12 /home/pasha/projects/bybit-history-downloader/data/raw/bybit/perpetual/linear/trade_ticks/BTCUSDT/BTCUSDT2026-08-12.csv.gz
2026-08-13 2026-08-13 /home/pasha/projects/bybit-history-downloader/data/raw/bybit/perpetual/linear/trade_ticks/BTCUSDT/BTCUSDT2026-08-13.csv.gz
2026-08-14 2026-08-14 /home/pasha/projects/bybit-history-downloader/data/raw/bybit/perpetual/linear/trade_ticks/BTCUSDT/BTCUSDT2026-08-14.csv.gz
2026-09-02 2026-09-02 /home/pasha/projects/bybit-history-downloader/data/raw/bybit/perpetual/linear/trade_ticks/BTCUSDT/BTCUSDT2026-09-02.csv.gz
2026-09-03 2026-09-03 /home/pasha/projects/bybit-history-downloader/data/raw/bybit/perpetual/linear/trade_tic

In [10]:
from marketforge.database.connection import connect
from marketforge.processing.metadata import (
    ProcessingMetadataResolver,
)


with connect() as conn:
    resolver = ProcessingMetadataResolver(conn)

    metadata = resolver.resolve(
        exchange="bybit",
        symbol="BTCUSDT",
        instrument_type="perpetual",
        market_category="linear",
        dataset="trade",
        profile="default",
    )

metadata

ProcessingMetadata(instrument=InstrumentMetadata(id=688, exchange='bybit', symbol='BTCUSDT', instrument_type='perpetual', market_category='linear', base_asset='BTC', quote_asset='USDT', settlement_asset='USDT', status='Trading', quantity_type='base', contract_value=None, contract_value_asset=None, tick_size=Decimal('0.10')), raw_format=RawFormatMetadata(id=10, format_code='BYBIT-T2', dataset='trade', instrument_type='perpetual', market_category='linear', container_format='csv', compression='gzip', record_format='rows', schema={'fields': [{'name': 'timestamp', 'type': 'decimal', 'unit': 'seconds', 'precision': 'up_to_4_fractional_digits', 'representation': 'unix_epoch'}, {'name': 'symbol', 'type': 'string'}, {'name': 'side', 'type': 'string'}, {'name': 'size', 'type': 'decimal'}, {'name': 'price', 'type': 'decimal'}, {'name': 'tickDirection', 'type': 'string'}, {'name': 'trdMatchID', 'type': 'uuid/string'}, {'name': 'grossValue', 'type': 'decimal'}, {'name': 'homeNotional', 'type': 'dec

In [11]:
from marketforge.processing.planning import (
    plan_process_dataset,
)


dataset_plan = plan_process_dataset(
    archives=selected,
    metadata=metadata,
)

dataset_plan

ProcessDatasetPlan(instrument_id=688, data_type='trade', start_timestamp_ns=1786320000000000000, end_timestamp_ns=1788480000000000000)

In [12]:
from datetime import datetime, timezone


start = datetime.fromtimestamp(
    dataset_plan.start_timestamp_ns / 1_000_000_000,
    tz=timezone.utc,
)

end = datetime.fromtimestamp(
    dataset_plan.end_timestamp_ns / 1_000_000_000,
    tz=timezone.utc,
)

print("Instrument ID:", dataset_plan.instrument_id)
print("Data type:    ", dataset_plan.data_type)
print("Start:        ", start)
print("End:          ", end)

Instrument ID: 688
Data type:     trade
Start:         2026-08-10 00:00:00+00:00
End:           2026-09-04 00:00:00+00:00


In [13]:
from marketforge.processing.planning import (
    write_process_job,
)


job_path = write_process_job(
    job_id=1,
    dataset_id=1,
    archives=selected,
    metadata=metadata,
    data_root=DATA_ROOT,
)

print("Processing Job")
print("=" * 80)
print(f"Path: {job_path}")
print()
print(job_path.read_text())

Processing Job
Path: /home/pasha/projects/bybit-history-downloader/data/.jobs/process.json

{
  "protocol_version": 1,
  "job_id": 1,
  "dataset_id": 1,
  "operation": "process",
  "tasks": [
    {
      "task_id": 0,
      "stream_id": "bybit:perpetual:linear:BTCUSDT:trade",
      "input_path": "/home/pasha/projects/bybit-history-downloader/data/raw/bybit/perpetual/linear/trade_ticks/BTCUSDT/BTCUSDT2026-08-10.csv.gz",
      "format_code": "BYBIT-T2",
      "exchange": "bybit",
      "instrument_id": 688,
      "symbol": "BTCUSDT",
      "raw_schema": {
        "fields": [
          {
            "name": "timestamp",
            "type": "decimal",
            "unit": "seconds",
            "precision": "up_to_4_fractional_digits",
            "representation": "unix_epoch"
          },
          {
            "name": "symbol",
            "type": "string"
          },
          {
            "name": "side",
            "type": "string"
          },
          {
            "name": "size